In [77]:
import os
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 100)

# ---- input ----
RECIPES_PKL = "recipes_candidates.pkl"
MAP_CSV = "ingredient_usda_map.csv"
PER100_PKL = "usda_nutrients_per100g.pkl"
PORTIONS_CSV = "ingredient_portions.csv"
UNIT_OVERRIDES_CSV = "manual_units.csv"

# ---- Settings ----
BIG_QTY = 30            
MIN_GAIN = 0.01          
MIN_RECIPES_CALIB = 20   
N_ROUNDS = 3
TEST_FRAC = 0.3
SEED = 42
REFIT_ON_ALL = True     
CAL_RANGE = (30, 2500)  
SERV_RANGE = (1, 48)    

NUTRIENTS = ["kcal", "protein_g", "fat_g", "carbs_g", "fiber_g", "calcium_mg", "iron_mg",
             "magnesium_mg", "potassium_mg", "sodium_mg", "zinc_mg", "vitc_mg",
             "folate_ug", "vitb6_mg", "vitb12_ug"]
UNIT_COLS = ["cup", "tbsp", "tsp", "piece", "oz"]

recipes = pd.read_pickle(RECIPES_PKL)
imap = pd.read_csv(MAP_CSV)
per100 = pd.read_pickle(PER100_PKL)
portions = pd.read_csv(PORTIONS_CSV)

NUTRIENTS = [c for c in NUTRIENTS if c in per100.columns]
print(recipes.shape, imap.shape, per100.shape, portions.shape)
print("nutrients available:", NUTRIENTS)

(46145, 20) (500, 17) (7793, 16) (1280, 6)
nutrients available: ['kcal', 'protein_g', 'fat_g', 'carbs_g', 'fiber_g', 'calcium_mg', 'iron_mg', 'magnesium_mg', 'potassium_mg', 'sodium_mg', 'zinc_mg', 'vitc_mg', 'folate_ug', 'vitb6_mg', 'vitb12_ug']


In [78]:
import re
import pandas as pd

RAW_RECIPES_CSV = "/Users/sarah/Desktop/Final Project/my-project/data/raw/recipes.csv"
REMOVE_LARD = True
REMOVE_GELATIN = False

PORK_WORDS = ["pork", "bacon", "ham", "sausage", "pepperoni", "prosciutto", "pancetta", "salami",
              "chorizo", "hot dog", "hotdog", "frankfurter", "kielbasa", "bratwurst", "mortadella",
              "capicola", "guanciale", "andouille", "bologna", "lardon", "spam", "chicharron"]
PORK_RE = re.compile(r"\b(?:" + "|".join(re.escape(w) for w in PORK_WORDS) + r")s?\b")
NOT_RE = re.compile(r"\b(?:turkey|chicken|vegetarian|veggie|soy|vegan|meatless|tofu|beef|lamb|seitan)\s+"
                    r"(?:bacon|sausage|ham|pepperoni|salami|hot dogs?|bologna|frankfurters?)s?\b")
PORK_FAT_RE = re.compile(r"\b(?:bacon|pork|ham|sausage)\s+(?:grease|drippings?|fat|renderings?)\b|\bsalt pork\b|\bham hocks?\b")
LARD_RE = re.compile(r"\blard\b")
GELATIN_RE = re.compile(r"\bgelatin(?:e)?\b")


def has_match(text, rx):
    if not isinstance(text, str):
        return False
    return rx.search(NOT_RE.sub(" ", text.lower())) is not None


def hit(series, rx):
    return pd.Series([has_match(t, rx) for t in series], index=series.index)


n_before = len(recipes)
rc = recipes.copy()

steps = pd.read_csv(RAW_RECIPES_CSV, usecols=["RecipeId", "RecipeInstructions"])
steps = steps[steps["RecipeId"].isin(rc["RecipeId"])].drop_duplicates("RecipeId")
rc = rc.merge(steps, on="RecipeId", how="left")

ing_txt = rc["parts_norm"].apply(lambda parts: " ".join(parts))
no = pd.Series(False, index=rc.index)

reasons = pd.DataFrame({
    "ingredients": hit(ing_txt, PORK_RE),
    "name": hit(rc["Name"], PORK_RE),
    "pork_fat_in_steps": hit(rc["RecipeInstructions"], PORK_FAT_RE),
    "lard": (hit(ing_txt, LARD_RE) | hit(rc["RecipeInstructions"], LARD_RE)) if REMOVE_LARD else no,
    "gelatin": (hit(ing_txt, GELATIN_RE) | hit(rc["RecipeInstructions"], GELATIN_RE)) if REMOVE_GELATIN else no,
})
pork_any = reasons.any(axis=1)

print(reasons.sum())
print(f"removed: {int(pork_any.sum()):,} of {n_before:,} ({pork_any.mean()*100:.1f}%)")

cols = list(reasons.columns)
removed = rc.loc[pork_any, ["RecipeId", "Name"]].copy()
removed["reasons"] = [",".join(c for c, v in zip(cols, row) if v) for row in reasons.loc[pork_any].to_numpy()]
removed.to_csv("removed_pork_recipes.csv", index=False, encoding="utf-8-sig")

recipes = recipes[~pork_any.to_numpy()].reset_index(drop=True)
print(f"remaining candidates: {len(recipes):,}")

ingredients          2570
name                 1430
pork_fat_in_steps     286
lard                   48
gelatin                 0
dtype: int64
removed: 2,851 of 46,145 (6.2%)
remaining candidates: 43,294


In [79]:
ing = imap[["ingredient", "fdc_id", "usda_description", "source", "confidence", "rank"]].copy()
ing = ing.dropna(subset=["fdc_id"]).drop_duplicates("ingredient")
ing["fdc_id"] = ing["fdc_id"].astype(int)
ing = ing.merge(per100[NUTRIENTS].reset_index(), on="fdc_id", how="left")
ing = ing.dropna(subset=["kcal"]).reset_index(drop=True)      
print(f"mapped ingredients with kcal: {len(ing)}")

p = portions.dropna(subset=["gram_weight"]).copy()
p["amount"] = pd.to_numeric(p["amount"], errors="coerce").fillna(1).replace(0, 1)
p["gpu"] = p["gram_weight"] / p["amount"]
p["label"] = p["portion_label"].fillna("").astype(str).str.lower().str.strip()

SKIP = ("serving", "nlea", "package", "container", "packet", "can ", "bottle", "jar", "cubic", "pat ", "dash", "drop", "scoop")

def unit_class(label):
    if label.startswith("tsp"):
        return "tsp"
    if label.startswith("tbsp"):
        return "tbsp"
    if label.startswith("cup"):
        return "cup"
    if label.startswith("fl oz"):
        return "floz"
    if label == "oz" or label.startswith("oz "):
        return "oz"
    if label == "" or any(s in label for s in SKIP):
        return None
    return "piece"

p["uc"] = p["label"].apply(unit_class)
p = p.dropna(subset=["uc"]).copy()
p["exact"] = -(p["label"] == p["uc"]).astype(int)          
p = p.sort_values(["ingredient", "uc", "exact", "seq_num"])
first = p.groupby(["ingredient", "uc"]).first().reset_index()

U = first.pivot(index="ingredient", columns="uc", values="gpu")
for c in ["cup", "tbsp", "tsp", "floz", "oz", "piece"]:
    if c not in U.columns:
        U[c] = np.nan
U = U.reindex(ing["ingredient"])

U["cup"] = U["cup"].fillna(U["tbsp"] * 16).fillna(U["tsp"] * 48).fillna(U["floz"] * 8)
U["tbsp"] = U["tbsp"].fillna(U["cup"] / 16)
U["tsp"] = U["tsp"].fillna(U["tbsp"] / 3)
U["oz"] = U["oz"].fillna(28.3495)

piece_label = first[first["uc"] == "piece"].set_index("ingredient")["label"]
print("\nNumber of ingredients that have each unit:")
print(U[UNIT_COLS].notna().sum())

mapped ingredients with kcal: 489

عدد المكونات اللي عندها كل وحدة:
uc
cup      396
tbsp     396
tsp      396
piece    183
oz       489
dtype: int64


In [80]:
ok_set = set(ing["ingredient"])
ing_index = {name: i for i, name in enumerate(ing["ingredient"])}

def recipe_ok(parts, qs):
    return all(x in ok_set for x in parts) and not any(np.isnan(q) or q <= 0 for q in qs)

mask_ok = np.array([recipe_ok(a, b) for a, b in zip(recipes["parts_norm"], recipes["quants_num"])])
dropped = recipes[~mask_ok]
R = recipes[mask_ok].reset_index(drop=True)
print(f"valid recipes: {len(R):,} / {len(recipes):,} | dropped (unmapped ingredient or bad quantity): {len(dropped):,}")

lost = Counter()
for parts in dropped["parts_norm"]:
    for x in set(parts):
        if x not in ok_set:
            lost[x] += 1
unmapped = pd.DataFrame(lost.most_common(), columns=["ingredient", "recipes_lost"])
unmapped.to_csv("unmapped_ingredients.csv", index=False, encoding="utf-8-sig")
unmapped.head(25)

valid recipes: 42,736 / 43,294 | dropped (unmapped ingredient or bad quantity): 558


,ingredient,recipes_lost
0,bourbon,135
1,fettuccine,79
2,panko breadcrumb,77
3,baguette,67
4,gorgonzola,49
5,mirin,45
6,cognac,42
7,kahlua,36
8,penne,36


In [81]:
desc_by_ing = ing.set_index("ingredient")["usda_description"].to_dict()
SPICE_KEYS = ("spices", "leavening", "salt, table", "extract", "seasoning", "yeast", "baking")


from collections import defaultdict
_qs = defaultdict(list)
for parts, qs in zip(R["parts_norm"], R["quants_num"]):
    for x, q in zip(parts, qs):
        _qs[x].append(q)


def count_like(name):
    if pd.isna(U.loc[name, "piece"]):
        return False
    q = np.array(_qs.get(name, []), float)
    q = q[q < BIG_QTY]
    return len(q) >= 30 and float(np.mean(q % 1 != 0)) < 0.15 and float(np.median(q)) <= 6


def prior_unit(name):
    row = U.loc[name]
    d = str(desc_by_ing.get(name, "")).lower()
    if any(k in d for k in SPICE_KEYS) and pd.notna(row["tsp"]):
        return "tsp"
    if count_like(name):
        return "piece"
    for u in ["cup", "piece", "tbsp", "tsp"]:
        if pd.notna(row[u]):
            return u
    return "oz"

unit_choice = np.array([UNIT_COLS.index(prior_unit(n)) for n in ing["ingredient"]])
locked = np.zeros(len(ing), dtype=bool)


def set_unit(ingredient, unit):
    assert unit in UNIT_COLS, f"Unit must be one of {UNIT_COLS}"
    ov = pd.read_csv(UNIT_OVERRIDES_CSV) if os.path.exists(UNIT_OVERRIDES_CSV) else pd.DataFrame(columns=["ingredient", "unit"])
    ov = pd.concat([ov[ov["ingredient"] != ingredient], pd.DataFrame([{"ingredient": ingredient, "unit": unit}])], ignore_index=True)
    ov.to_csv(UNIT_OVERRIDES_CSV, index=False, encoding="utf-8-sig")
    print("saved:", ingredient, "->", unit)

if os.path.exists(UNIT_OVERRIDES_CSV):
    for r in pd.read_csv(UNIT_OVERRIDES_CSV).itertuples():
        if r.ingredient in ing_index and r.unit in UNIT_COLS and pd.notna(U.loc[r.ingredient, r.unit]):
            unit_choice[ing_index[r.ingredient]] = UNIT_COLS.index(r.unit)
            locked[ing_index[r.ingredient]] = True
print(f"locked units from {UNIT_OVERRIDES_CSV}: {int(locked.sum())}")
print(pd.Series([UNIT_COLS[i] for i in unit_choice]).value_counts().to_dict())

locked units from manual_units.csv: 62
{'cup': 301, 'piece': 88, 'tsp': 64, 'oz': 36}


In [82]:
SPICE_ANY = ("leavening", "salt, table", "extract", "yeast")
rows = []
for name in ing["ingredient"]:
    d = str(desc_by_ing.get(name, "")).lower()
    if (d.startswith("spices") or any(k in d for k in SPICE_ANY)) and pd.notna(U.loc[name, "tsp"]):
        rows.append({"ingredient": name, "unit": "tsp"})
new = pd.DataFrame(rows, columns=["ingredient", "unit"])
old = pd.read_csv(UNIT_OVERRIDES_CSV) if os.path.exists(UNIT_OVERRIDES_CSV) else pd.DataFrame(columns=["ingredient", "unit"])
new = new[~new["ingredient"].isin(old["ingredient"])]
pd.concat([old, new], ignore_index=True).to_csv(UNIT_OVERRIDES_CSV, index=False, encoding="utf-8-sig")
print(f"locked to tsp: {len(new)} | existing manual kept: {len(old)}")
print(sorted(new["ingredient"]))

locked to tsp: 0 | existing manual kept: 62
[]


In [83]:
n_rec, n_ing = len(R), len(ing)
ridx, iidx, qty = [], [], []
for r, (parts, qs) in enumerate(zip(R["parts_norm"], R["quants_num"])):
    for x, q in zip(parts, qs):
        ridx.append(r)
        iidx.append(ing_index[x])
        qty.append(q)
ridx = np.array(ridx, dtype=np.int64)
iidx = np.array(iidx, dtype=np.int64)
qty = np.array(qty, dtype=float)
print(f"rows: {len(ridx):,}")

G = U.loc[ing["ingredient"], UNIT_COLS].to_numpy(float)       
K = ing["kcal"].to_numpy(float) / 100                        
Nmat = ing[NUTRIENTS].to_numpy(float) / 100                    

serv = R["RecipeServings"].to_numpy(float)
cal = R["Calories"].to_numpy(float)
has_serv = R["has_servings"].to_numpy(bool) & (serv >= SERV_RANGE[0]) & (serv <= SERV_RANGE[1])
cal_mask = has_serv & (cal >= CAL_RANGE[0]) & (cal <= CAL_RANGE[1])
print(f"recipes with usable servings+calories for calibration/validation: {int(cal_mask.sum()):,} ({cal_mask.mean()*100:.1f}%)")


def grams_rows(uc):
    gu = G[iidx, uc[iidx]]
    return np.where(qty >= BIG_QTY, qty, qty * gu)


def evaluate(T, mask):
    e = np.log((T[mask] / serv[mask] + 1) / (cal[mask] + 1))
    a = np.abs(e)
    return {"n": int(mask.sum()),
            "median_ratio": round(float(np.exp(np.median(e))), 3),
            "median_abs_log_err": round(float(np.median(a)), 3),
            "within_25%": round(float((a <= np.log(1.25)).mean()), 3),
            "within_50%": round(float((a <= np.log(1.5)).mean()), 3)}


T0 = np.bincount(ridx, weights=grams_rows(unit_choice) * K[iidx], minlength=n_rec)
print("Prior units, all calibration recipes:", evaluate(T0, cal_mask))

rows: 331,688
recipes with usable servings+calories for calibration/validation: 24,324 (56.9%)
Prior units, all calibration recipes: {'n': 24324, 'median_ratio': 1.872, 'median_abs_log_err': 0.725, 'within_25%': 0.28, 'within_50%': 0.377}


In [84]:
order = np.argsort(iidx, kind="stable")
sorted_i = iidx[order]
starts = np.searchsorted(sorted_i, np.arange(n_ing))
ends = np.searchsorted(sorted_i, np.arange(n_ing), side="right")
freq = ends - starts
ing_order = np.argsort(-freq)


def calibrate(mask, uc0, verbose=True):
    uc = uc0.copy()
    kc = grams_rows(uc) * K[iidx]
    T = np.bincount(ridx, weights=kc, minlength=n_rec)
    for rnd in range(N_ROUNDS):
        changed = 0
        for i in ing_order:
            if locked[i] or freq[i] == 0:
                continue
            rows = order[starts[i]:ends[i]]
            recs = ridx[rows]
            uniq, inv = np.unique(recs, return_inverse=True)
            m = mask[uniq]
            if m.sum() < MIN_RECIPES_CALIB:
                continue
            q = qty[rows]
            big = q >= BIG_QTY
            cur = np.bincount(inv, weights=kc[rows], minlength=len(uniq))
            base = T[uniq] - cur
            su, cu = serv[uniq][m], cal[uniq][m]
            res = {}
            for u in range(len(UNIT_COLS)):
                if np.isnan(G[i, u]):
                    continue
                new = np.bincount(inv, weights=np.where(big, q, q * G[i, u]) * K[i], minlength=len(uniq))
                Tn = base + new
                loss = float(np.median(np.abs(np.log((Tn[m] / su + 1) / (cu + 1)))))
                res[u] = (loss, new)
            if not res or uc[i] not in res:
                continue
            u_best = min(res, key=lambda u: res[u][0])
            if u_best != uc[i] and res[u_best][0] <= res[uc[i]][0] - MIN_GAIN:
                uc[i] = u_best
                T[uniq] = base + res[u_best][1]
                kc[rows] = np.where(big, q, q * G[i, u_best]) * K[i]
                changed += 1
        if verbose:
            print(f"round {rnd + 1}: {changed} ingredients changed unit")
        if changed == 0:
            break
    return uc


rng = np.random.default_rng(SEED)
idx = np.where(cal_mask)[0]
rng.shuffle(idx)
n_test = int(len(idx) * TEST_FRAC)
test_mask = np.zeros(n_rec, dtype=bool)
train_mask = np.zeros(n_rec, dtype=bool)
test_mask[idx[:n_test]] = True
train_mask[idx[n_test:]] = True

uc_train = calibrate(train_mask, unit_choice)

def T_of(uc):
    return np.bincount(ridx, weights=grams_rows(uc) * K[iidx], minlength=n_rec)

res = pd.DataFrame({
    "prior | train": evaluate(T0, train_mask), "calibrated | train": evaluate(T_of(uc_train), train_mask),
    "prior | TEST": evaluate(T0, test_mask), "calibrated | TEST": evaluate(T_of(uc_train), test_mask),
}).T
res

round 1: 126 ingredients changed unit
round 2: 80 ingredients changed unit
round 3: 13 ingredients changed unit


,n,median_ratio,median_abs_log_err,within_25%,within_50%
prior | train,17027.0,1.864,0.719,0.282,0.378
calibrated | train,17027.0,1.013,0.368,0.347,0.536
prior | TEST,7297.0,1.902,0.737,0.276,0.372
calibrated | TEST,7297.0,1.028,0.386,0.339,0.517


In [85]:
unit_final = calibrate(cal_mask, uc_train) if REFIT_ON_ALL else uc_train
T_final = T_of(unit_final)
print("Final units, ALL calibration recipes (in-sample):", evaluate(T_final, cal_mask))

grams_final = grams_rows(unit_final)
du = pd.DataFrame({
    "ingredient": ing["ingredient"],
    "usda_description": ing["usda_description"],
    "unit": [UNIT_COLS[i] for i in unit_final],
    "g_per_unit": [G[k, unit_final[k]] for k in range(n_ing)],
    "prior_unit": [UNIT_COLS[i] for i in unit_choice],
    "locked": locked,
    "n_recipes": freq,
})
du["changed_by_calibration"] = du["unit"] != du["prior_unit"]
du = du.sort_values("n_recipes", ascending=False).reset_index(drop=True)
du.to_csv("default_units.csv", index=False, encoding="utf-8-sig")
print(f"units changed by calibration: {int(du['changed_by_calibration'].sum())} of {len(du)}")
du.head(40)

round 1: 60 ingredients changed unit
round 2: 16 ingredients changed unit
round 3: 18 ingredients changed unit
Final units, ALL calibration recipes (in-sample): {'n': 24324, 'median_ratio': 1.013, 'median_abs_log_err': 0.368, 'within_25%': 0.349, 'within_50%': 0.536}
units changed by calibration: 164 of 489


,ingredient,usda_description,unit,g_per_unit,prior_unit,locked,n_recipes,changed_by_calibration
0,salt,"Salt, table",tsp,6.0000,tsp,True,23257,False
1,butter,"Butter, salted",tbsp,14.2000,cup,False,20111,True
2,sugar,"Sugars, granulated",cup,200.0000,cup,False,19323,False
3,all purpose flour,"Wheat flour, white, all-purpose, enriched, bleached",cup,125.0000,cup,False,16776,False
4,egg,"Egg, whole, raw, fresh",cup,243.0000,piece,False,15082,True
5,vanilla extract,Vanilla extract,tsp,4.2000,tsp,True,10058,False
6,water,"Beverages, water, tap, drinking",cup,236.8000,cup,False,8126,False
7,baking powder,"Leavening agents, baking powder, double-acting, sodium a...",tsp,4.6000,tsp,True,7779,False
8,milk,"Milk, whole, 3.25% milkfat, with added vitamin D",cup,244.0000,cup,False,7778,False
9,brown sugar,"Sugars, brown",cup,220.0000,cup,False,7697,False


In [86]:
implied = T_final / np.maximum(cal, 1)

known = cal_mask & (serv <= 12)
diff = np.abs(np.round(implied[known]) - serv[known])
print(f"recipes with known servings (<=12): {int(known.sum()):,}")
print(f"estimated servings == given: {(diff == 0).mean()*100:.1f}% | within +-1: {(diff <= 1).mean()*100:.1f}%")
print(f"median(implied / given) = {np.median(implied[known] / serv[known]):.2f}")

serv_final = np.where(has_serv, serv, np.clip(np.round(implied), 1, 24))
serv_source = np.where(has_serv, "given", "estimated")
print(pd.Series(serv_source).value_counts().to_dict())

recipes with known servings (<=12): 20,354
estimated servings == given: 16.1% | within +-1: 38.9%
median(implied / given) = 1.05
{'given': 25049, 'estimated': 17687}


In [87]:
tot = {}
for k, name in enumerate(NUTRIENTS):
    tot[name] = np.bincount(ridx, weights=grams_final * np.nan_to_num(Nmat[iidx, k]), minlength=n_rec)

tot_kcal = tot["kcal"]
scale = np.where((tot_kcal > 0) & (cal > 0), cal / np.maximum(tot_kcal, 1e-9), np.nan)

out = R[["RecipeId", "Name", "RecipeCategory", "TotalTime_min", "parts_norm", "quants_num",
         "n_ingredients", "Calories"]].copy()
out["servings_given"] = np.where(has_serv, serv, np.nan)
out["servings_final"] = serv_final
out["servings_source"] = serv_source
out["kcal_ratio"] = np.where(has_serv, (tot_kcal / np.where(has_serv, serv, 1)) / np.maximum(cal, 1), np.nan)

for name in NUTRIENTS:
    out[f"{name}_ps"] = tot[name] * scale
    out[f"{name}_ps_direct"] = np.where(has_serv, tot[name] / np.where(has_serv, serv, 1), np.nan)

for name in ("iron_mg", "vitc_mg", "magnesium_mg"):
    if name in NUTRIENTS:
        k = NUTRIENTS.index(name)
        miss = (np.isnan(Nmat[iidx, k]) & (grams_final >= 1)).astype(float)
        out[f"missing_{name}"] = np.bincount(ridx, weights=miss, minlength=n_rec).astype(int)

r_ = out["kcal_ratio"]
out["quality"] = np.select(
    [r_.isna(), (r_ >= 1 / 1.3) & (r_ <= 1.3), (r_ >= 1 / 1.6) & (r_ <= 1.6)],
    ["unverified", "good", "fair"], default="poor")
print(out["quality"].value_counts().to_dict())

if "iron_mg" in NUTRIENTS:
    both = out["iron_mg_ps_direct"].notna() & (out["iron_mg_ps"] > 0)
    ratio = out.loc[both, "iron_mg_ps_direct"] / out.loc[both, "iron_mg_ps"]
    print(f"\nIron: direct / anchored  median={ratio.median():.2f} | within +-30%: {((ratio > 1/1.3) & (ratio < 1.3)).mean()*100:.1f}%")

out[["Name", "Calories", "servings_final", "servings_source", "kcal_ratio", "quality",
     "iron_mg_ps", "vitc_mg_ps", "magnesium_mg_ps", "protein_g_ps"]].head(10)

{'unverified': 17687, 'poor': 10590, 'good': 9782, 'fair': 4677}

Iron: direct / anchored  median=1.02 | within +-30%: 39.1%


,Name,Calories,servings_final,servings_source,kcal_ratio,quality,iron_mg_ps,vitc_mg_ps,magnesium_mg_ps,protein_g_ps
0,Betty Crocker's Southwestern Guacamole Dip,415.9,4.0,given,0.799981,good,1.971146,38.083470,74.139125,6.668852
1,Buttermilk Pie,395.9,8.0,given,1.421206,fair,1.943883,0.222966,12.365534,6.843960
2,California Chilled Salsa,21.0,10.0,given,4.596714,poor,0.072206,11.214641,2.033844,0.170328
3,All Purpose Quick Mix With 28 Variations,529.3,3.0,estimated,NaN,unverified,6.659217,0.000000,56.072324,14.172420
4,Butter Madeira Cake,373.8,8.0,given,1.323997,fair,2.304770,0.000000,15.848595,11.705170
5,Butter Dips,64.1,24.0,estimated,NaN,unverified,0.479742,0.000000,2.901680,1.175878
6,Blue Stuffed Mushrooms,246.3,4.0,given,12.131305,poor,1.128409,2.061279,45.547201,4.525822
7,Borscht,631.8,2.0,estimated,NaN,unverified,3.585893,102.348851,66.060111,4.163998
8,Carrot Ginger Biscuits,127.0,12.0,given,0.836022,good,1.115394,0.995547,13.600747,2.670309
9,Blueberry Scones,277.3,8.0,given,2.172585,poor,1.263093,3.021401,7.886401,5.236136


In [88]:
log_err = np.log((T_final / np.where(has_serv, serv, 1) + 1) / (cal + 1))
bad_rec = cal_mask & (np.abs(log_err) > np.log(2))
overall_bad = bad_rec[cal_mask].mean()
print(f"bad recipes among calibration set: {overall_bad*100:.1f}%")

rows = []
for i in range(n_ing):
    if freq[i] == 0:
        continue
    recs = np.unique(ridx[order[starts[i]:ends[i]]])
    recs = recs[cal_mask[recs]]
    if len(recs) < 25:
        continue
    nb_ = int(bad_rec[recs].sum())
    rows.append({"ingredient": ing.loc[i, "ingredient"], "usda_description": ing.loc[i, "usda_description"],
                 "unit": UNIT_COLS[unit_final[i]], "n_cal_recipes": len(recs), "n_bad": nb_,
                 "bad_rate": round(nb_ / len(recs), 3), "mean_log_err": round(float(log_err[recs].mean()), 3),
                 "confidence": ing.loc[i, "confidence"]})
susp = pd.DataFrame(rows)
susp = susp[(susp["bad_rate"] >= max(2 * overall_bad, 0.3))].sort_values("n_bad", ascending=False)
susp.to_csv("suspect_ingredients.csv", index=False, encoding="utf-8-sig")
print(f"{len(susp)} suspect ingredients")
susp.head(30)

bad recipes among calibration set: 28.4%
2 suspect ingredients


,ingredient,usda_description,unit,n_cal_recipes,n_bad,bad_rate,mean_log_err,confidence
260,chicken wing,"Chicken, broilers or fryers, wing, meat and skin, raw",piece,71,43,0.606,-0.561,medium
434,wine vinegar,"Vinegar, red wine",cup,33,20,0.606,-0.302,high


In [89]:
out.to_pickle("recipes_nutrients.pkl")
print(out.shape)

summary = pd.DataFrame({
    "step": ["candidate recipes (notebook 2)", "all ingredients mapped to USDA (with kcal)", "servings given (usable)",
             "quality = good", "quality = fair", "quality = poor", "quality = unverified"],
    "n_recipes": [len(recipes), len(R), int(has_serv.sum()),
                  int((out["quality"] == "good").sum()), int((out["quality"] == "fair").sum()),
                  int((out["quality"] == "poor").sum()), int((out["quality"] == "unverified").sum())],
})
summary

(42736, 46)


,step,n_recipes
0,candidate recipes (notebook 2),43294
1,all ingredients mapped to USDA (with kcal),42736
2,servings given (usable),25049
3,quality = good,9782
4,quality = fair,4677
5,quality = poor,10590
6,quality = unverified,17687
